# PixiJoy Flow (M4) — Kaggle GPU (quality)

**Settings (right panel):** Accelerator → **GPU T4** (or P100) → Internet **ON** → Save.

Kaggle free tier ≈ **30 GPU hours / week**, then resets.

**Resume across sessions:** after a run, download `dist/*.ckpt.keras` + `train_state.json`,
or *Save Version* (Save & Run All) and add the output as a Dataset Input next time under
`/kaggle/input/.../`. Put files into `dist/` before train.

Pin TFLite only if verify prints **OK** / **SHIP_OK**.


In [ ]:
!nvidia-smi || true
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TF", tf.__version__, "GPUs", gpus, flush=True)
assert gpus, 'Enable GPU: Notebook settings → Accelerator → GPU T4 → Save → Restart'


In [ ]:
import os, sys, urllib.request, zipfile, shutil
from pathlib import Path

ROOT = Path('/kaggle/working/pixijoy_flow')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
(ROOT / 'dist').mkdir(exist_ok=True)
(ROOT / 'data').mkdir(exist_ok=True)
print('cwd', ROOT, flush=True)

# Optional: copy prior ckpt from a Kaggle Dataset Input
for inp in Path('/kaggle/input').glob('*'):
    for name in (f'{keras_name}', f'{Path(keras_name).stem}.ckpt.keras', 'train_state.json'):
        src = next(inp.rglob(name), None)
        if src and src.is_file():
            dst = ROOT / 'dist' / src.name
            shutil.copy2(src, dst)
            print('restored from input', src, '->', dst, flush=True)

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/flow_lab.zip'
LAB_ZIP_VERSION = '20260928e'
urllib.request.urlretrieve(ZIP_URL + f'?v={LAB_ZIP_VERSION}', 'flow_lab.zip')
with zipfile.ZipFile('flow_lab.zip') as z:
    z.extractall('.')
assert Path('train.py').is_file()
print('lab zip OK', flush=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path

def run_stream(cmd):
    print('STREAM', ' '.join(map(str, cmd)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    print(f'[exit {rc}]', flush=True)
    return rc

rc = run_stream([sys.executable, '-u', 'prepare_quality_data.py', '--data', './data'])
assert rc == 0
assert (Path('data') / 'READY').is_file()
print('data READY ok', flush=True)


In [ ]:
import os, hashlib, subprocess, sys
from pathlib import Path

OUT = Path('dist') / 'flow.keras'
ckpt = OUT.with_suffix('.ckpt.keras')

def run_stream(cmd):
    print('STREAM', ' '.join(map(str, cmd)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

cmd = [sys.executable, '-u', 'train.py', '--quality', '--out', str(OUT)]
if ckpt.is_file():
    cmd.append('--resume')
    print('will resume', ckpt, flush=True)
rc = run_stream(cmd)
assert rc == 0
rc = run_stream([sys.executable, '-u', 'export_tflite.py', '--keras', str(OUT), '--out', 'dist/flow.tflite'])
assert rc == 0
rc = run_stream([sys.executable, '-u', 'verify_contract.py', '--quality', '--data', './data', '--model', 'dist/flow.tflite'])
assert rc == 0, 'verify failed — do NOT pin'
p = Path('dist/flow.tflite')
print('SHIP_OK', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest(), flush=True)
print('Download dist/ from Output, or Save Version to keep GPU session artifacts.', flush=True)
